# Feature infrastructure: example run (task D)

Demonstrates the `features/` package: a common `Feature.calculate(...)` interface, a
leak-safe `FeatureContext`, a `FeatureSet` pipeline (sample + streaming), and validators
(**no NaN / no inf / aligned / no forward-looking**, the last via two independent checks).
Unit tests: `tests/` (50 tests). The heavy data is
handled out-of-core, this notebook draws a reproducible 2M-trade sample per symbol and keeps
peak memory ~10 GB.

In [1]:
import polars as pl
from features import FeatureContext, FeatureSet, DEFAULT_FEATURES, load_frames
from features.base import Feature

pl.Config.set_tbl_rows(12); pl.Config.set_tbl_width_chars(160)

pl.DataFrame({"feature": [f.name for f in DEFAULT_FEATURES],
              "lookback_s": [f.lookback_us // 1_000_000 for f in DEFAULT_FEATURES],
              "inputs": [", ".join(f.inputs) for f in DEFAULT_FEATURES]})

feature,lookback_s,inputs
str,i64,str
"""liq_notional_30s""",30,"""liq_combined"""
"""liq_notional_120s""",120,"""liq_combined"""
"""liq_event_count_30s""",30,"""liq_combined"""
"""liq_side_imbalance_30s""",30,"""liq_combined"""
"""exchange_2_liq_notional_30s""",30,"""liq_exchange_2"""
"""liq_velocity""",120,"""liq_combined"""
"""time_since_liq_s""",0,"""liq_combined"""
"""bbo_spread_bps""",0,"""bbo"""
"""bbo_imbalance""",0,"""bbo"""


## Example run on BTC & ETH

For each symbol: load a 2M-trade reservoir sample + full BBO/liquidations, build the
leak-safe context (Exchange 2 +200 ms applied inside), compute the feature matrix, and validate.
Trades before the first quote (+5 s for the trailing-return lookback) are excluded, they
can't be quoted or marked out before the order book exists.

In [2]:
def run_symbol(sym, sample=2_000_000):
    trades, bbo, lb, lby = load_frames(sym, trades_sample=sample, seed=42)
    trades = trades.filter(pl.col("timestamp") >= bbo["timestamp"].min() + 5_000_000)
    ctx = FeatureContext.from_frames(trades, bbo, lb, lby)
    del bbo
    fs = FeatureSet(DEFAULT_FEATURES)
    matrix = fs.run(ctx)
    report = fs.validate(ctx, n_probe=128)
    feats = [f.name for f in DEFAULT_FEATURES]
    stats = (matrix.select(feats).describe()
            .filter(pl.col("statistic").is_in(["mean", "std", "min", "max"])))
    import numpy as np
    fwd = ctx.asof("mid", offset_us=30_000_000)
    markout = -ctx.trade_s * (fwd - ctx.trade_price) / ctx.trade_price * 1e4
    return {"sym": sym, "n": matrix.height, "shape": matrix.shape, "matrix": matrix,
            "markout": np.asarray(markout),
            "report": report.select("feature", "n_nan", "n_inf", "aligned",
                                    "lookahead_ok", "tracked_ok", "ok"),
            "head": matrix.head(6), "stats": stats, "all_ok": bool(report["ok"].all())}

btc = run_symbol("btc")
print(f"BTC: {btc['n']:,} trades, matrix {btc['shape']}, validation all-pass = {btc['all_ok']}")
btc["report"]

BTC: 1,999,994 trades, matrix (1999994, 11), validation all-pass = True


feature,n_nan,n_inf,aligned,lookahead_ok,tracked_ok,ok
str,i64,i64,bool,bool,bool,bool
"""liq_notional_30s""",0,0,true,true,true,true
"""liq_notional_120s""",0,0,true,true,true,true
"""liq_event_count_30s""",0,0,true,true,true,true
"""liq_side_imbalance_30s""",0,0,true,true,true,true
"""exchange_2_liq_notional_30s""",0,0,true,true,true,true
"""liq_velocity""",0,0,true,true,true,true
"""time_since_liq_s""",0,0,true,true,true,true
"""bbo_spread_bps""",0,0,true,true,true,true
"""bbo_imbalance""",0,0,true,true,true,true


In [3]:
print("BTC feature matrix (head):")
btc["head"]

BTC feature matrix (head):


timestamp,liq_notional_30s,liq_notional_120s,liq_event_count_30s,liq_side_imbalance_30s,exchange_2_liq_notional_30s,liq_velocity,time_since_liq_s,bbo_spread_bps,bbo_imbalance,mid_ret_5s
i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
1764547207262000,4852.9772,4852.9772,4.0,-1.0,3234.1524,0.999794,0.161,0.011078,0.553071,-3.864893
1764547208653000,48319.7704,48319.7704,11.0,-1.0,46700.9456,0.999979,0.494,0.011079,-0.530433,-2.746764
1764547208653000,48319.7704,48319.7704,11.0,-1.0,46700.9456,0.999979,0.494,0.011079,-0.530433,-2.746764
1764547212137000,52636.156,52636.156,12.0,-1.0,46700.9456,0.999981,3.071,0.011074,0.39224,3.855308
1764547212137000,52636.156,52636.156,12.0,-1.0,46700.9456,0.999981,3.071,0.011074,0.39224,3.855308
1764547212138000,52636.156,52636.156,12.0,-1.0,46700.9456,0.999981,3.072,0.011074,0.39224,3.855308


In [4]:
eth = run_symbol("eth")
print(f"ETH: {eth['n']:,} trades, matrix {eth['shape']}, validation all-pass = {eth['all_ok']}")
eth["report"]

ETH: 1,999,998 trades, matrix (1999998, 11), validation all-pass = True


feature,n_nan,n_inf,aligned,lookahead_ok,tracked_ok,ok
str,i64,i64,bool,bool,bool,bool
"""liq_notional_30s""",0,0,true,true,true,true
"""liq_notional_120s""",0,0,true,true,true,true
"""liq_event_count_30s""",0,0,true,true,true,true
"""liq_side_imbalance_30s""",0,0,true,true,true,true
"""exchange_2_liq_notional_30s""",0,0,true,true,true,true
"""liq_velocity""",0,0,true,true,true,true
"""time_since_liq_s""",0,0,true,true,true,true
"""bbo_spread_bps""",0,0,true,true,true,true
"""bbo_imbalance""",0,0,true,true,true,true


Validation result: all 10 features pass on both symbols, finite, aligned to the trade
timestamps, and no forward-looking. The last property is checked two ways: `lookahead_ok` is
the differential probe (rebuild a sample of trades from only the data available by t and require
the same value) and `tracked_ok` is exhaustive (run the feature with the context recording, per
trade, the latest source timestamp it touched, and require it to be `<= t` on every trade).

## The no-lookahead validators catch deliberate leaks

A feature is leak-safe only if it reads the context causally. Here a deliberately-broken
feature peeks at the **future** quote (mid at t+30 s). Both checks flag it: the differential
`check_no_lookahead` (a sample) and the exhaustive `check_no_lookahead_tracked` (every trade).
They are complementary, the differential one also catches a feature that reads an event at
*exactly* t while bypassing the public window methods that the tracker instruments.

In [5]:
from features.validate import check_no_lookahead, check_no_lookahead_tracked
from features import library as lib

class LeakyForward(Feature):
    name = "leaky_forward_mid"
    def calculate(self, ctx):
        return ctx.asof("mid", offset_us=30_000_000)

S = 1_000_000
trades = pl.DataFrame({"timestamp": [t*S for t in [50, 100, 150]], "ticker": ["x"]*3,
                       "side": ["buy","sell","buy"], "price": [100.0]*3, "amount": [1.0]*3})
bbo = pl.DataFrame({"timestamp": [t*S for t in [10,55,90,105,140,200]], "ticker": ["x"]*6,
                    "bid_price": [100,101,100,102,100,103], "bid_amount": [1.0]*6,
                    "ask_price": [101,102,101,103,101,104], "ask_amount": [1.0]*6})
lb = pl.DataFrame({"timestamp": [t*S for t in [45,95,145]], "ticker": ["x"]*3,
                   "side": ["sell","buy","sell"], "price": [100.0]*3, "amount": [10.0]*3})
lby = pl.DataFrame({"timestamp": [t*S for t in [48,98]], "ticker": ["x"]*2,
                    "side": ["buy","sell"], "price": [100.0]*2, "amount": [5.0]*2})
ctx = FeatureContext.from_frames(trades, bbo, lb, lby)

for f in (lib.LiqNotional(30), LeakyForward()):
    print(f"{f.name:18s} differential={check_no_lookahead(f, ctx)['ok']!s:5s} "
          f"tracked={check_no_lookahead_tracked(f, ctx)['ok']!s:5s} "
          f"(tracked violations={check_no_lookahead_tracked(f, ctx)['n_violations']})")

liq_notional_30s   differential=True  tracked=True  (tracked violations=0)
leaky_forward_mid  differential=False tracked=False (tracked violations=2)


## Ranking features against forward markout

The optional `features.analyze` suite scores each feature against a target (here the 30 s
signed maker markout in bps). Spearman IC is rank correlation (sign + monotone strength);
KS separation is the distributional gap between the toxic (markout < 0) and benign fills.
Mutual information and an out-of-sample tree R² are also available (they need scikit-learn).
The ICs are small and this is in-sample on one window, so read it as a sanity-check ordering,
not a performance claim; the cross-regime evidence is in EDA §4.

In [6]:
from features.analyze import SpearmanIC, KSSeparation, analyze_features

import numpy as np
feat_cols = [f.name for f in DEFAULT_FEATURES]
toxic = np.where(np.isfinite(btc["markout"]), btc["markout"] < 0, np.nan)
tbl = analyze_features(btc["matrix"].select(feat_cols), btc["markout"], [SpearmanIC()], feat_cols)
ks = analyze_features(btc["matrix"].select(feat_cols), toxic, [KSSeparation()], feat_cols)
(tbl.join(ks, on="feature")
    .with_columns(abs_ic=pl.col("spearman_ic").abs())
    .sort("abs_ic", descending=True).drop("abs_ic"))

feature,spearman_ic,ks_separation
str,f64,f64
"""liq_event_count_30s""",0.015232,0.036353
"""exchange_2_liq_notional_30s""",0.014791,0.030124
"""liq_notional_30s""",0.014731,0.036543
"""time_since_liq_s""",-0.014648,0.037457
"""liq_velocity""",0.013146,0.036353
"""liq_notional_120s""",0.011274,0.038818
"""bbo_imbalance""",0.006708,0.012485
"""liq_side_imbalance_30s""",-0.002829,0.020131
"""mid_ret_5s""",-0.001883,0.018464


## Scaling to full data

The same feature classes and validators run over all 402M / 706M trades via streaming,
liquidations + BBO are prepared once (BBO ~1.6 GB in RAM) and trades flow through in 20M-row
batches, so a windowed feature always sees the complete past stream (no batch-boundary halo):

```python
FeatureSet(DEFAULT_FEATURES).run_streaming(
    "../liquidation_task/data/exchange_1_trades/perp_btcusdt.parquet",
    "../liquidation_task/data/exchange_1_booktickers/perp_btcusdt.parquet",
    "../liquidation_task/data/exchange_1_liquidations/perp_btcusdt.parquet",
    "../liquidation_task/data/exchange_2_liquidations/btcusdt.parquet",
    out_path="features_btc.parquet")
```

Adding a feature is one small `Feature` subclass appended to `DEFAULT_FEATURES`; the
pipeline and validators pick it up automatically. The library also ships extras outside the
default set, e.g. `LiqStrength` (signed `log1p` of net liquidation flow, a heavy-tail-robust
transform).